In [ ]:
!pip install -q google-genai==1.66.0 db-dtypes

In [ ]:
## Import libraries
import time
import json
import requests
import pandas as pd

from datetime import datetime, timezone
from google.cloud import bigquery
from google.cloud import storage
from google import genai

In [ ]:
# config
PROJECT_ID = "qwiklabs-gcp-00-871084f9eb9e"
DATASET_ID = "aero_alerts"
AIRPORTS_TABLE = "airports"
ALERTS_TABLE = "airport_weather_alerts"

BUCKET_NAME = f"{PROJECT_ID}-aero-alerts"
SOURCE_CSV_URI = "gs://labs.roitraining.com/data-to-ai-workshop/airports.csv"
DEST_BLOB_NAME = "airports/airports.csv"
CSV_GCS_URI = f"gs://{BUCKET_NAME}/{DEST_BLOB_NAME}"

BQ_LOCATION = "US"
BUCKET_LOCATION = "US"
VERTEX_LOCATION = "us-central1"

In [ ]:
# Create Google Cloud and Gemini clients
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=VERTEX_LOCATION
)

In [ ]:
## Create BigQuery dataset
dataset_id_full = f"{PROJECT_ID}.{DATASET_ID}"

dataset = bigquery.Dataset(dataset_id_full)
dataset.location = BQ_LOCATION

try:
    bq_client.get_dataset(dataset_id_full)
    print(f"Dataset already exists: {dataset_id_full}")
except Exception:
    bq_client.create_dataset(dataset, exists_ok=True)
    print(f"Created dataset: {dataset_id_full}")

Created dataset: qwiklabs-gcp-00-871084f9eb9e.aero_alerts


In [ ]:
## Create Cloud Storage bucket if needed
def create_bucket_if_needed(bucket_name, location="US"):
    bucket = storage_client.bucket(bucket_name)
    if bucket.exists():
        print(f"Bucket already exists: {bucket_name}")
        return bucket

    bucket = storage.Bucket(storage_client, name=bucket_name)
    bucket.location = location
    bucket = storage_client.create_bucket(bucket)
    print(f"Created bucket: {bucket_name}")
    return bucket

bucket = create_bucket_if_needed(BUCKET_NAME, BUCKET_LOCATION)

/tmp/ipykernel_7677/4130880150.py:9: DeprecationWarning: Assignment to 'Bucket.location' is deprecated, as it is only valid before the bucket is created. Instead, pass the location to `Bucket.create`.
  bucket.location = location


Created bucket: qwiklabs-gcp-00-871084f9eb9e-aero-alerts


In [ ]:
## Copy airport CSV into your bucket
def copy_gcs_file(source_uri, destination_bucket_name, destination_blob_name):
    source_uri = source_uri.replace("gs://", "")
    source_bucket_name = source_uri.split("/")[0]
    source_blob_name = "/".join(source_uri.split("/")[1:])

    source_bucket = storage_client.bucket(source_bucket_name)
    source_blob = source_bucket.blob(source_blob_name)

    destination_bucket = storage_client.bucket(destination_bucket_name)
    copied_blob = source_bucket.copy_blob(
        source_blob,
        destination_bucket,
        destination_blob_name
    )

    print(f"Copied {source_blob_name} to gs://{destination_bucket_name}/{destination_blob_name}")
    return copied_blob

copy_gcs_file(SOURCE_CSV_URI, BUCKET_NAME, DEST_BLOB_NAME)

Copied data-to-ai-workshop/airports.csv to gs://qwiklabs-gcp-00-871084f9eb9e-aero-alerts/airports/airports.csv


<Blob: qwiklabs-gcp-00-871084f9eb9e-aero-alerts, airports/airports.csv, 1780497229171122>

In [ ]:
## Verify CSV exists in your bucket
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(DEST_BLOB_NAME)

if blob.exists():
    print(f"File exists: gs://{BUCKET_NAME}/{DEST_BLOB_NAME}")
else:
    print("File not found in destination bucket")

File exists: gs://qwiklabs-gcp-00-871084f9eb9e-aero-alerts/airports/airports.csv


In [ ]:
## Load airport CSV from Cloud Storage into BigQuery
airports_table_id = f"{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TABLE}"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE"
)

load_job = bq_client.load_table_from_uri(
    CSV_GCS_URI,
    airports_table_id,
    job_config=job_config
)

load_job.result()
print(f"Loaded CSV into {airports_table_id}")

Loaded CSV into qwiklabs-gcp-00-871084f9eb9e.aero_alerts.airports


In [ ]:
## Preview airport data
preview_query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TABLE}`
LIMIT 10
"""

airports_preview_df = bq_client.query(preview_query).to_dataframe()
airports_preview_df

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11,NA,US,US-PA,Bensalem,False,None,None,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,None,None
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435,NA,US,US-KS,Leoti,False,None,None,00AA,00AA,None,None,None
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450,NA,US,US-AK,Anchor Point,False,None,None,00AK,00AK,None,None,None
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820,NA,US,US-AL,Harvest,False,None,None,00AL,00AL,None,None,None
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80,NA,US,US-AK,King Salmon,False,None,None,00AN,00AN,None,None,None
5,322127,00AS,small_airport,Fulton Airport,34.942803,-97.818019,1100,NA,US,US-OK,Alex,False,None,None,00AS,00AS,None,None,None
6,6527,00AZ,small_airport,Cordes Airport,34.305599,-112.165001,3810,NA,US,US-AZ,Cordes,False,None,None,00AZ,00AZ,None,None,None
7,6528,00CA,small_airport,Goldstone (GTS) Airport,35.354740,-116.885329,3038,NA,US,US-CA,Barstow,False,None,None,00CA,00CA,None,https://en.wikipedia.org/wiki/Goldstone_Gts_Ai...,None
8,324424,00CL,small_airport,Williams Ag Airport,39.427188,-121.763427,87,NA,US,US-CA,Biggs,False,None,None,00CL,00CL,None,None,None
9,322658,00CN,heliport,Kitchen Creek Helibase Heliport,32.727374,-116.459742,3350,NA,US,US-CA,Pine Valley,False,None,None,00CN,00CN,None,None,None


In [ ]:
## Filter large US airports
large_airports_query = f"""
SELECT
  ident,
  type,
  name,
  iso_country,
  municipality,
  latitude_deg,
  longitude_deg
FROM `{PROJECT_ID}.{DATASET_ID}.{AIRPORTS_TABLE}`
WHERE type = 'large_airport'
  AND iso_country = 'US'
  AND latitude_deg IS NOT NULL
  AND longitude_deg IS NOT NULL
ORDER BY name
"""

airports_df = bq_client.query(large_airports_query).to_dataframe()
print(f"Total large US airports: {len(airports_df)}")
airports_df.head()

Total large US airports: 71


,ident,type,name,iso_country,municipality,latitude_deg,longitude_deg
0,KABQ,large_airport,Albuquerque International Sunport,US,Albuquerque,35.039976,-106.608925
1,KAUS,large_airport,Austin Bergstrom International Airport,US,Austin,30.197535,-97.662015
2,KBWI,large_airport,Baltimore/Washington International Thurgood Ma...,US,Baltimore,39.175400,-76.668297
3,KBDL,large_airport,Bradley International Airport,US,Hartford,41.938510,-72.688066
4,KBUF,large_airport,Buffalo Niagara International Airport,US,Buffalo,42.940498,-78.732201


In [ ]:
## Define National Weather Service API function
def get_extended_weather_forecast(lat, lon):
    headers = {
        "User-Agent": "MyWeatherApp (shivani.maheshwarm@zionclouds.com)",
        "Accept": "application/geo+json"
    }

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        response = requests.get(points_url, headers=headers, timeout=30)

        if response.status_code != 200:
            return None, f"Points API error: {response.status_code}"

        points_data = response.json()
        forecast_url = points_data.get("properties", {}).get("forecast")

        if not forecast_url:
            return None, "Forecast URL not found in points response"

        forecast_response = requests.get(forecast_url, headers=headers, timeout=30)

        if forecast_response.status_code != 200:
            return None, f"Forecast API error: {forecast_response.status_code}"

        forecast_data = forecast_response.json()
        periods = forecast_data.get("properties", {}).get("periods", [])

        if not periods:
            return None, "No forecast periods found"

        extended_forecast = []
        for period in periods:
            extended_forecast.append({
                "name": period.get("name"),
                "startTime": period.get("startTime"),
                "temperature": period.get("temperature"),
                "temperatureUnit": period.get("temperatureUnit"),
                "windSpeed": period.get("windSpeed"),
                "windDirection": period.get("windDirection"),
                "shortForecast": period.get("shortForecast"),
                "detailedForecast": period.get("detailedForecast")
            })

        return extended_forecast, None

    except requests.exceptions.Timeout:
        return None, "Request timed out"
    except requests.exceptions.ConnectionError:
        return None, "Connection error"
    except requests.exceptions.RequestException as e:
        return None, f"Request failed: {str(e)}"
    except Exception as e:
        return None, f"Unexpected error: {str(e)}"

In [ ]:
## Test the weather API function
forecast, error = get_extended_weather_forecast(42.090098, -83.161499)

if error:
    print("Error:", error)
else:
    for period in forecast[:3]:
        print(period)

{'name': 'Today', 'startTime': '2026-06-03T09:00:00-04:00', 'temperature': 79, 'temperatureUnit': 'F', 'windSpeed': '2 to 9 mph', 'windDirection': 'E', 'shortForecast': 'Sunny', 'detailedForecast': 'Sunny, with a high near 79. East wind 2 to 9 mph.'}
{'name': 'Tonight', 'startTime': '2026-06-03T18:00:00-04:00', 'temperature': 57, 'temperatureUnit': 'F', 'windSpeed': '3 to 8 mph', 'windDirection': 'S', 'shortForecast': 'Mostly Clear', 'detailedForecast': 'Mostly clear, with a low around 57. South wind 3 to 8 mph.'}
{'name': 'Thursday', 'startTime': '2026-06-04T06:00:00-04:00', 'temperature': 84, 'temperatureUnit': 'F', 'windSpeed': '3 to 10 mph', 'windDirection': 'SSW', 'shortForecast': 'Mostly Sunny', 'detailedForecast': 'Mostly sunny, with a high near 84. South southwest wind 3 to 10 mph.'}


In [ ]:
## Define Gemini alert generation function
def generate_airport_alert(airport_name, city, forecast_periods):
    try:
        forecast_text = "\n".join([
            (
                f"- {p.get('name')}: {p.get('temperature')} {p.get('temperatureUnit')}, "
                f"{p.get('shortForecast')}, Wind {p.get('windSpeed')} {p.get('windDirection')}. "
                f"Details: {p.get('detailedForecast')}"
            )
            for p in forecast_periods[:4]
        ])

        prompt = f"""
You are creating a public-facing airport weather advisory.

Airport: {airport_name}
City: {city}

Forecast:
{forecast_text}

Write a concise weather advisory for travelers, pilots, and airport staff.

Rules:
- Use plain language.
- Keep it factual and safety-focused.
- 2 to 4 sentences only.
- Mention likely operational or passenger impact if relevant.
- Avoid unnecessary jargon.
"""

        response = genai_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        if not response or not getattr(response, "text", None):
            return None, "Gemini returned an empty response"

        return response.text.strip(), None

    except Exception as e:
        return None, f"Gemini generation failed: {str(e)}"

In [ ]:
## Test Gemini alert generation
if error is None and forecast:
    alert_text, gemini_error = generate_airport_alert(
        airport_name="Detroit Metropolitan Wayne County Airport",
        city="Detroit",
        forecast_periods=forecast
    )

    if gemini_error:
        print("Gemini error:", gemini_error)
    else:
        print(alert_text)

Detroit Metropolitan Wayne County Airport anticipates excellent weather conditions for today and tomorrow. Expect sunny to mostly clear skies, light winds, and good visibility, ensuring smooth operations for travelers and pilots. No significant weather impacts or delays are anticipated.


In [ ]:
## Process airports and generate alerts
results = []

sample_airports_df = airports_df.copy()

for _, row in sample_airports_df.iterrows():
    ident = row["ident"]
    airport_name = row["name"]
    city = row["municipality"]
    lat = row["latitude_deg"]
    lon = row["longitude_deg"]

    print(f"Processing {airport_name} ({ident})...")

    forecast_periods, weather_error = get_extended_weather_forecast(lat, lon)

    if weather_error:
        results.append({
            "ident": ident,
            "airport_name": airport_name,
            "city": city,
            "latitude_deg": lat,
            "longitude_deg": lon,
            "status": "FAILED",
            "error_message": weather_error,
            "forecast_json": None,
            "alert_text": None,
            "processed_at": datetime.now(timezone.utc)
        })
        continue

    alert_text, gemini_error = generate_airport_alert(
        airport_name=airport_name,
        city=city,
        forecast_periods=forecast_periods
    )

    if gemini_error:
        results.append({
            "ident": ident,
            "airport_name": airport_name,
            "city": city,
            "latitude_deg": lat,
            "longitude_deg": lon,
            "status": "FAILED",
            "error_message": gemini_error,
            "forecast_json": json.dumps(forecast_periods),
            "alert_text": None,
            "processed_at": datetime.now(timezone.utc)
        })
        continue

    results.append({
        "ident": ident,
        "airport_name": airport_name,
        "city": city,
        "latitude_deg": lat,
        "longitude_deg": lon,
        "status": "SUCCESS",
        "error_message": None,
        "forecast_json": json.dumps(forecast_periods),
        "alert_text": alert_text,
        "processed_at": datetime.now(timezone.utc)
    })

    time.sleep(1)


Processing Albuquerque International Sunport (KABQ)...
Processing Austin Bergstrom International Airport (KAUS)...
Processing Baltimore/Washington International Thurgood Marshall Airport (KBWI)...
Processing Bradley International Airport (KBDL)...
Processing Buffalo Niagara International Airport (KBUF)...
Processing Charlotte Douglas International Airport (KCLT)...
Processing Chicago Midway International Airport (KMDW)...
Processing Chicago O'Hare International Airport (KORD)...
Processing Cincinnati Northern Kentucky International Airport (KCVG)...
Processing Cleveland Hopkins International Airport (KCLE)...
Processing Dallas Fort Worth International Airport (KDFW)...
Processing Daniel K Inouye International Airport (PHNL)...
Processing Denver International Airport (KDEN)...
Processing Detroit Metropolitan Wayne County Airport (KDTW)...
Processing Eppley Airfield (KOMA)...
Processing Fort Lauderdale Hollywood International Airport (KFLL)...
Processing General Mitchell International Ai

In [ ]:
## Convert results to a DataFrame
alerts_df = pd.DataFrame(results)
print(alerts_df.shape)
alerts_df.head()

(71, 10)


,ident,airport_name,city,latitude_deg,longitude_deg,status,error_message,forecast_json,alert_text,processed_at
0,KABQ,Albuquerque International Sunport,Albuquerque,35.039976,-106.608925,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T08...",**Albuquerque International Sunport Weather Ad...,2026-06-03 14:55:58.057535+00:00
1,KAUS,Austin Bergstrom International Airport,Austin,30.197535,-97.662015,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",Austin Bergstrom International Airport anticip...,2026-06-03 14:56:06.574864+00:00
2,KBWI,Baltimore/Washington International Thurgood Ma...,Baltimore,39.175400,-76.668297,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",BWI Airport expects sunny to mostly clear skie...,2026-06-03 14:56:24.996920+00:00
3,KBDL,Bradley International Airport,Hartford,41.938510,-72.688066,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Weather Advisory for Bradley International A...,2026-06-03 14:56:32.452978+00:00
4,KBUF,Buffalo Niagara International Airport,Buffalo,42.940498,-78.732201,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Buffalo Niagara International Airport Weathe...,2026-06-03 14:56:39.305089+00:00


In [ ]:
## Extract summary forecast fields
def extract_first_period_value(forecast_json_str, field_name):
    try:
        forecast = json.loads(forecast_json_str) if forecast_json_str else []
        if forecast and field_name in forecast[0]:
            return forecast[0].get(field_name)
        return None
    except Exception:
        return None

alerts_df["forecast_name"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "name"))
alerts_df["temperature"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "temperature"))
alerts_df["temperature_unit"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "temperatureUnit"))
alerts_df["wind_speed"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "windSpeed"))
alerts_df["wind_direction"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "windDirection"))
alerts_df["short_forecast"] = alerts_df["forecast_json"].apply(lambda x: extract_first_period_value(x, "shortForecast"))

alerts_df.head()

,ident,airport_name,city,latitude_deg,longitude_deg,status,error_message,forecast_json,alert_text,processed_at,forecast_name,temperature,temperature_unit,wind_speed,wind_direction,short_forecast
0,KABQ,Albuquerque International Sunport,Albuquerque,35.039976,-106.608925,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T08...",**Albuquerque International Sunport Weather Ad...,2026-06-03 14:55:58.057535+00:00,Today,83,F,5 to 10 mph,SE,Partly Sunny then Chance Showers And Thunderst...
1,KAUS,Austin Bergstrom International Airport,Austin,30.197535,-97.662015,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",Austin Bergstrom International Airport anticip...,2026-06-03 14:56:06.574864+00:00,Today,90,F,0 to 10 mph,ENE,Partly Sunny then Chance Showers And Thunderst...
2,KBWI,Baltimore/Washington International Thurgood Ma...,Baltimore,39.175400,-76.668297,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",BWI Airport expects sunny to mostly clear skie...,2026-06-03 14:56:24.996920+00:00,Today,83,F,7 mph,N,Sunny
3,KBDL,Bradley International Airport,Hartford,41.938510,-72.688066,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Weather Advisory for Bradley International A...,2026-06-03 14:56:32.452978+00:00,Today,84,F,1 to 5 mph,NW,Sunny
4,KBUF,Buffalo Niagara International Airport,Buffalo,42.940498,-78.732201,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Buffalo Niagara International Airport Weathe...,2026-06-03 14:56:39.305089+00:00,Today,78,F,5 to 12 mph,SW,Sunny


In [ ]:
## Load generated alerts into BigQuery
alerts_table_id = f"{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

load_job = bq_client.load_table_from_dataframe(
    alerts_df,
    alerts_table_id,
    job_config=job_config
)

load_job.result()
print(f"Loaded alerts into {alerts_table_id}")

Loaded alerts into qwiklabs-gcp-00-871084f9eb9e.aero_alerts.airport_weather_alerts


In [ ]:
## Preview the alerts table
alerts_preview_query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}`
ORDER BY processed_at DESC
LIMIT 20
"""

alerts_preview_df = bq_client.query(alerts_preview_query).to_dataframe()
alerts_preview_df

,ident,airport_name,city,latitude_deg,longitude_deg,status,error_message,forecast_json,alert_text,processed_at,forecast_name,temperature,temperature_unit,wind_speed,wind_direction,short_forecast
0,KOKC,Will Rogers World Airport,Oklahoma City,35.393388,-97.598248,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",**Will Rogers World Airport Weather Advisory**...,2026-06-03 15:05:31.265585+00:00,Today,87,F,9 to 13 mph,SE,Chance Showers And Thunderstorms
1,KIAD,Washington Dulles International Airport,Dulles,38.944500,-77.455803,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Washington Dulles International Airport (IAD...,2026-06-03 15:05:25.043300+00:00,Today,82,F,9 mph,N,Sunny
2,KTUL,Tulsa International Airport,Tulsa,36.198399,-95.888100,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T09...",**Tulsa International Airport (TUL) Weather Ad...,2026-06-03 15:05:17.899588+00:00,Today,84,F,10 mph,SE,Slight Chance Showers And Thunderstorms
3,KPVD,Theodore Francis Green State Airport,Warwick,41.725038,-71.425668,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",**Theodore Francis Green State Airport (Warwic...,2026-06-03 15:05:12.540512+00:00,Today,82,F,1 to 10 mph,SW,Sunny
4,PANC,Ted Stevens Anchorage International Airport,Anchorage,61.179004,-149.992561,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",Ted Stevens Anchorage International Airport an...,2026-06-03 15:05:02.730394+00:00,Today,72,F,0 to 5 mph,W,Mostly Sunny
5,KTPA,Tampa International Airport,Tampa,27.975500,-82.533203,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",Travelers and staff at Tampa International Air...,2026-06-03 15:04:56.886475+00:00,Today,84,F,8 to 15 mph,ENE,Chance Showers And Thunderstorms
6,KSYR,Syracuse Hancock International Airport,Syracuse,43.111198,-76.106300,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T10...",Syracuse Hancock International Airport anticip...,2026-06-03 15:04:48.891709+00:00,Today,80,F,6 to 9 mph,NW,Sunny
7,KSTL,St. Louis Lambert International Airport,St Louis,38.748697,-90.370003,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T08...","For St. Louis Lambert International Airport, e...",2026-06-03 15:04:40.839115+00:00,Today,83,F,5 to 9 mph,SE,Sunny
8,KRSW,Southwest Florida International Airport,Fort Myers,26.536200,-81.755203,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...",**Airport Weather Advisory: Southwest Florida ...,2026-06-03 15:04:31.783488+00:00,Today,83,F,3 to 13 mph,ENE,Chance Showers And Thunderstorms
9,KSEA,Seattle–Tacoma International Airport,Seattle,47.447943,-122.310276,SUCCESS,None,"[{""name"": ""Today"", ""startTime"": ""2026-06-03T06...","For Seattle–Tacoma International Airport, anti...",2026-06-03 15:04:22.269658+00:00,Today,69,F,10 mph,SW,Mostly Cloudy


In [ ]:
## Add geography column for Looker Studio map
geo_query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}` AS
SELECT
  *,
  ST_GEOGPOINT(longitude_deg, latitude_deg) AS location
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}`
"""

bq_client.query(geo_query).result()
print("Added geography column to alerts table")

Added geography column to alerts table


In [ ]:
## Validate final table
final_query = f"""
SELECT
  ident,
  airport_name,
  city,
  status,
  short_forecast,
  alert_text,
  processed_at,
  location
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}`
LIMIT 20
"""

final_df = bq_client.query(final_query).to_dataframe()
final_df.head()

,ident,airport_name,city,status,short_forecast,alert_text,processed_at,location
0,KIAH,George Bush Intercontinental Houston Airport,Houston,SUCCESS,Chance Showers And Thunderstorms,**Houston George Bush Intercontinental Airport...,2026-06-03 14:58:19.160225+00:00,POINT(-95.3414001464844 29.9843997955322)
1,KTPA,Tampa International Airport,Tampa,SUCCESS,Chance Showers And Thunderstorms,Travelers and staff at Tampa International Air...,2026-06-03 15:04:56.886475+00:00,POINT(-82.533203 27.9755)
2,KSRQ,Sarasota Bradenton International Airport,Sarasota/Bradenton,SUCCESS,Chance Showers And Thunderstorms,**Sarasota Bradenton International Airport (SR...,2026-06-03 15:04:05.447566+00:00,POINT(-82.554359 27.394631)
3,KSAT,San Antonio International Airport,San Antonio,SUCCESS,Chance Showers And Thunderstorms,**San Antonio International Airport (SAT) Advi...,2026-06-03 15:03:37.524518+00:00,POINT(-98.469803 29.533701)
4,KRSW,Southwest Florida International Airport,Fort Myers,SUCCESS,Chance Showers And Thunderstorms,**Airport Weather Advisory: Southwest Florida ...,2026-06-03 15:04:31.783488+00:00,POINT(-81.7552032470703 26.5361995697021)


In [ ]:
## Summary of processed airports
summary_query = f"""
SELECT
  status,
  COUNT(*) AS airport_count
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}`
GROUP BY status
ORDER BY airport_count DESC
"""

summary_df = bq_client.query(summary_query).to_dataframe()
summary_df

,status,airport_count
0,SUCCESS,71


In [ ]:
final_query = f"""
SELECT
  ident,
  airport_name,
  city,
  status,
  short_forecast,
  alert_text,
  processed_at,
  location
FROM `{PROJECT_ID}.{DATASET_ID}.{ALERTS_TABLE}`
LIMIT 20
"""

final_df = bq_client.query(final_query).to_dataframe()
final_df.head()

,ident,airport_name,city,status,short_forecast,alert_text,processed_at,location
0,KIAH,George Bush Intercontinental Houston Airport,Houston,SUCCESS,Chance Showers And Thunderstorms,**Houston George Bush Intercontinental Airport...,2026-06-03 14:58:19.160225+00:00,POINT(-95.3414001464844 29.9843997955322)
1,KTPA,Tampa International Airport,Tampa,SUCCESS,Chance Showers And Thunderstorms,Travelers and staff at Tampa International Air...,2026-06-03 15:04:56.886475+00:00,POINT(-82.533203 27.9755)
2,KSRQ,Sarasota Bradenton International Airport,Sarasota/Bradenton,SUCCESS,Chance Showers And Thunderstorms,**Sarasota Bradenton International Airport (SR...,2026-06-03 15:04:05.447566+00:00,POINT(-82.554359 27.394631)
3,KSAT,San Antonio International Airport,San Antonio,SUCCESS,Chance Showers And Thunderstorms,**San Antonio International Airport (SAT) Advi...,2026-06-03 15:03:37.524518+00:00,POINT(-98.469803 29.533701)
4,KRSW,Southwest Florida International Airport,Fort Myers,SUCCESS,Chance Showers And Thunderstorms,**Airport Weather Advisory: Southwest Florida ...,2026-06-03 15:04:31.783488+00:00,POINT(-81.7552032470703 26.5361995697021)
